# 第五轮讨论：基于论文与代码精读的架构深化

（等待 Idea subagent 补读论文和代码后填入讨论内容）

## 5.1 精读发现：GET-Zero × T(R,O) 的真正互补点

**用户反馈**：要求先精读 GET-Zero（正文 1-6 页）、T(R,O) Grasp（正文 1-8 页 + 附录 11-12 页）以及双方核心代码，再基于精读结果继续讨论架构设计；同时前四轮已收敛为：只做手型泛化、joint-level tokenization、两阶段主干 `Cross-Attn(dynamic×static) -> Self-Attn + Graph Bias`，并保留方案 A / B。

**分析**：

### 一、论文精读：GET-Zero 的强项是“离散拓扑先验”，不是“完整 embodiment 表征”

1. **GET 的主公式很克制**。论文第 3 页（Sec. III-B, Fig. 2）把图信息直接加在 attention score 上：
$$
A_{ij} = \frac{Q_iK_j^T}{\sqrt{d}} + s_{\phi^{SPD}(i,j)} + p_{\phi^P(i,j)} + c_{\phi^C(i,j)}.
$$
这说明 GET 真正验证的是：**joint token + graph bias** 足以把运动学拓扑塞进 policy backbone。

2. **Token 里的 fixed embodiment 信息其实并不丰富**。论文第 2-4 页（Sec. III-A / IV-C）写得很清楚：固定局部量主要是 URDF 导出的 joint 位置/旋转等；没有 link geometry、joint limits、motor strength、friction 这些跨 hand family 真正敏感的量。

3. **GET 的训练范式不是“直接多手型 RL”**。论文第 4 页（Fig. 3, Sec. IV-B/C）是 `44 个 embodiment-specific PPO experts -> 每个 7h demonstration -> BC distillation`。所以它的 zero-shot 结论，很大一部分建立在 **teacher-student pipeline** 上，而不是端到端多构型 RL 上。

4. **第 5 页 Table I 给了一个很关键的信号**：`ET+SE+SL` 和 `ET+PE+SE+SL` 在 `New Graph` 上几乎持平（10.04 vs 10.07），说明 **parent-child directed bias 不是主要增益来源**；真正起作用的是更一般的 spatial graph encoding + self-modeling。

5. **作者自己在第 6 页把边界说得很明白**：现模型“不太可能”零样本转到全新手型，因为没有编码 `joint limit ranges, motor strength, friction properties or finger shape`。这句话其实直接给了我们切入点：**GET 的瓶颈不在 graph transformer 本身，而在 embodiment 编码过弱**。

### 二、论文精读：T(R,O) Grasp 的强项是“连续几何关系”，不是“它是 diffusion”

1. 第 3-4 页（Eq. 2-7）最重要的不是 diffusion，而是它怎么定义图：
   - object node = patch center + scale + geometry token
   - link node = link 几何编码 + 当前 SE(3) pose
   - OR / RR edge = relative SE(3)

2. 第 4-5 页（Eq. 12）真正值得借的是：
$$
V_{ij} = h_V(x_i, x_j, e_{ij}),
$$
也就是 **边特征不只改 attention score，而是直接改 message content**。这和 GET 的 Graphormer-style bias 是本质不同的。

3. 第 6-8 页和附录 11-12 页也透露了工程边界：object 用 $P=25$ 个 patches，link 节点 zero-pad 到 25，inference 用 $M=20$ 个 DDIM steps。它已经比 D(R,O) 轻很多，但对逐步 policy 来说仍然偏重。换句话说：**T(R,O) 最值得借的是 relation encoding，不是把 diffusion 主干整个搬过来**。

### 三、代码精读：GET-Zero 的图信息确实只进了 score，没有进 Value

| 证据 | 代码位置 | 说明 |
|---|---|---|
| padding + mask 处理变长 DoF | `get_zero/get_zero/distill/models/embodiment_transformer.py:189` | GET 的 batch 混合多手型时，靠 `src_key_padding_mask` 解决 |
| 额外的“geometry/property embedding”其实只覆盖离散结构属性 | `get_zero/get_zero/distill/models/embodiment_transformer.py:62-67` | 这里只给 degree / parent / child / child-link-id 做 embedding；看不到 link shape / joint limits 这类 richer embodiment 描述 |
| token 真正进入主干前只做一次线性映射 | `get_zero/get_zero/distill/models/embodiment_transformer.py:76, 231` | `token_embedding = nn.Linear(...)`，说明 stronger embodiment prior 主要不在 token mixer，而在输入和 attention bias |
| Graph bias 构造 | `get_zero/get_zero/distill/models/embodiment_attention.py:382-476` | `_sa_block` 里把 SPD / parent / child / edge embedding 组装成 `attention_bias` |
| bias 注入的位置 | `get_zero/get_zero/distill/models/embodiment_attention.py:911-930` | `attention_reweighting` 只在 softmax 前/后加到 attention weights，`V` 本身不带 edge conditioning |
| RL 适配器很薄 | `get_zero/get_zero/rl/models/embodiment_transformer.py:62-68, 93-105` | RL 侧只是包一层预训练 student，固定 `policy` / `policy_forward_kinematics` 头做推理 |
| 可视化工具只画 bias 表 | `get_zero/get_zero/distill/models/vis_embodiment_transformer.py:65-94` | 进一步说明 GET 最想看的就是 learned graph bias，而不是 richer relation message |

**这一点很关键**：GET-Zero 不是“不会用图”，而是它用图的方式非常保守——**离散拓扑偏置 + joint token**。

### 四、代码精读：T(R,O) 的 relation message 是连续且显式的

| 证据 | 代码位置 | 说明 |
|---|---|---|
| link 几何编码 = BPS + centroid + scale | `TRO-Grasp/model/tro_graph.py:98, 138-178` | `construct_bps()` 先做 BPS，再过 `link_token_encoder` |
| 训练图中每步都重算 OR / RR 相对位姿 | `TRO-Grasp/model/tro_graph.py:434-624`，尤其 `:615, :623` | edge 不是静态 ID，而是随 pose 更新的连续 SE(3) 关系 |
| OR Value 直接吃 `object node + robot node + edge` | `TRO-Grasp/model/denoiser.py:166, 255` | 说明 object→robot 消息内容由边调制 |
| RR Value 直接吃 `self node + neighbor node + edge` | `TRO-Grasp/model/denoiser.py:188, 294` | 说明 robot→robot 消息内容也由边调制 |
| 多层输出做 Dense 聚合 | `TRO-Grasp/model/denoiser.py:418, 571` | 更像在堆关系层，而不是只学一个 bias 表 |

所以 TRO-Grasp 的真正启发是：**连续相对几何最好进入 message，而不只是进入 score。**

### 五、把两边放在一起后，真正冒出来的架构问题变了

前四轮里我们一直在说“Graph Transformer vs Relational Attention”。精读完后，我觉得更精确的说法应该是：

> **真正要拍板的，不是“要不要图”，而是“连续几何关系到底只进 bias，还是进一步进 message / value”。**

因为 hand-family 泛化下，至少有两类先验要同时存在：

1. **离散拓扑先验**：SPD、parent、child —— 这是 GET 已验证有效的部分；
2. **连续几何先验**：rest-pose relative SE(3)、link geometry、joint limits —— 这是 GET 明确缺失、而 TRO 强调的部分。

对应到我们当前已经收敛的两阶段主干，更自然的写法是：
$$
H^{(0)} = \mathrm{CrossAttn}(X^{dyn}, X^{stat}),
$$
$$
\alpha_{ij} \propto \exp\left(\frac{Q_iK_j^T}{\sqrt d} + b^{topo}_{ij} + b^{geom}_{ij}\right).
$$

但这里还有最后一个分叉：
- 若 $\tilde V_{ij} = V_j$，那就是 **Graphormer-style bias 增强版**；
- 若 $\tilde V_{ij} = h_\psi(h_i, h_j, e_{ij})$，那就是 **TRO-style relational message**。

### 六、与用户个人构思的对齐

你在 `mine.ipynb` 里已经提前定了三件事：
1. 动作空间坚持 joint space；
2. `DirectRLEnv` 先做快速原型；
3. 很难直接端到端 RL 训练 cross-embodiment 策略。

精读后看，这三点和文献证据其实是对齐的：
- `joint-space action` 更像 GET，不像 TRO diffusion；
- `原型先跑通` 更支持先做 policy backbone，而不是把 diffusion 图整个搬来；
- `端到端多手 RL 难` 则和 GET 采用 distillation pipeline 的事实一致。

**小结**：精读后的核心结论不是“GET 不行，TRO 更强”，而是：**GET 负责 online joint policy 的骨架，TRO 负责告诉我们‘连续几何关系不该只当离散 bias 的替代品’。当前真正需要拍板的是：连续几何关系只进 self-attention 的 bias，还是继续进 edge-conditioned message。**

**待确认**：下一步你更想把第 5 轮的架构收敛到哪一条线上：`rich static + bias-only`，还是 `rich static + edge-conditioned value`？
## 5.2 基于精读后，最自然的主干岔口

**用户反馈**：希望讨论方向由精读结果自然驱动，不预设方向；但讨论最终要落到可操作的架构决策上。

**分析**：

如果只做手型泛化、且沿用我们已经收敛的两阶段主干，那么现在最自然的三个实现版本其实是下面这张表：

| 版本 | Cross-Attn(dynamic×static) | Self-Attn 阶段 | 主要优点 | 主要风险 |
|---|---|---|---|---|
| **A. 保守版** | joint state 查询 static embodiment tokens（joint limits / rest pose / link geometry） | Graphormer-style：$b^{topo}_{ij} + g_\phi(e_{ij})$ 只改 score | 最贴近 GET；实现最稳；适合先验证 hand-family 泛化 | 连续几何只影响“看谁”，不影响“传什么” |
| **B. 关系增强版** | 同 A | Relational Self-Attn：$\tilde V_{ij}=h_\psi(h_i,h_j,e_{ij})$ | 方法差异更强；更像真正把 TRO 的长板接进 policy backbone | 实现更难，训练更可能不稳 |
| **C. 双轨版** | 同 A | 接口按 B 设计，但第一轮实验先跑 A，把 B 当 upgrade / ablation | 论文故事和工程路线都比较顺 | 需要一开始就把模块边界设计清楚 |

### 一、为什么我觉得这已经是“本轮最值得拍板的点”

因为前四轮里更上游的东西其实已经基本不摇了：
- token 粒度 = joint-level；
- 动作空间 = joint space；
- 主体骨架 = `Cross-Attn(dynamic×static) -> Self-Attn(graph-aware) -> per-joint head`；
- 目标 = 先做 hand-family generalization，不做 object generalization。

在这个前提下，再往下走，最影响后续实现和实验 story 的就是：

> **第一篇 paper 的主要贡献，究竟放在“更强的 embodiment input/static stream”，还是放在“更强的 relation message passing”。**

### 二、A 和 B 的论文叙事其实不一样

- **A 的 story**：
  > GET 的主干已经够好，真正短板在 embodiment 编码太弱；只要把 static stream 做强，并用 cross-attention 显式对齐动态状态与结构先验，就能跨出 LEAP family。

- **B 的 story**：
  > 仅靠 graph bias 还不够；跨 hand family 时，连续几何关系必须进入 message content，而不只是 attention score。

这两句话都成立，但它们对应的实验设计、消融重点、实现代价都不一样。

### 三、我现在的判断

如果你问“哪条更像第一篇 paper 的稳路线”，我会说：
- **A 更稳**，因为它已经足够构成一篇完整工作：`GET-style backbone + richer embodiment stream + hand-family benchmark`；
- **B 更强**，因为它更像真正的方法创新，而不是 feature completion。

所以现在最像样的选择，不是简单的 A 或 B 二选一，而是：

$$
\text{paper-1 primary claim} \in \{\text{stronger embodiment stream},\; \text{stronger relational message}\}.
$$

一旦这句话定了，后面的辅助 loss、训练范式、baseline 和实验优先级都会自然收缩。

**小结**：精读把讨论压缩成了一个很具体的选择：**我们是把贡献放在“更强的 embodiment token / static stream”，还是放在“更强的 relation message”。**

**待确认**：你更想把第一篇 paper 的主贡献落在哪一句 story 上？

## 5.2 基于精读后，最自然的主干岔口

**用户反馈**：希望讨论方向由精读结果自然驱动，不预设方向；但讨论最终要落到可操作的架构决策上。

**分析**：

如果只做手型泛化、且沿用我们已经收敛的两阶段主干，那么现在最自然的三个实现版本其实是下面这张表：

| 版本 | Cross-Attn(dynamic×static) | Self-Attn 阶段 | 主要优点 | 主要风险 |
|---|---|---|---|---|
| **A. 保守版** | joint state 查询 static embodiment tokens（joint limits / rest pose / link geometry） | Graphormer-style：$b^{topo}_{ij} + g_\phi(e_{ij})$ 只改 score | 最贴近 GET；实现最稳；适合先验证 hand-family 泛化 | 连续几何只影响“看谁”，不影响“传什么” |
| **B. 关系增强版** | 同 A | Relational Self-Attn：$\tilde V_{ij}=h_\psi(h_i,h_j,e_{ij})$ | 方法差异更强；更像真正把 TRO 的长板接进 policy backbone | 实现更难，训练更可能不稳 |
| **C. 双轨版** | 同 A | 接口按 B 设计，但第一轮实验先跑 A，把 B 当 upgrade / ablation | 论文故事和工程路线都比较顺 | 需要一开始就把模块边界设计清楚 |

### 一、为什么我觉得这已经是“本轮最值得拍板的点”

因为前四轮里更上游的东西其实已经基本不摇了：
- token 粒度 = joint-level；
- 动作空间 = joint space；
- 主体骨架 = `Cross-Attn(dynamic×static) -> Self-Attn(graph-aware) -> per-joint head`；
- 目标 = 先做 hand-family generalization，不做 object generalization。

在这个前提下，再往下走，最影响后续实现和实验 story 的就是：

> **第一篇 paper 的主要贡献，究竟放在“更强的 embodiment input/static stream”，还是放在“更强的 relation message passing”。**

### 二、A 和 B 的论文叙事其实不一样

- **A 的 story**：
  > GET 的主干已经够好，真正短板在 embodiment 编码太弱；只要把 static stream 做强，并用 cross-attention 显式对齐动态状态与结构先验，就能跨出 LEAP family。

- **B 的 story**：
  > 仅靠 graph bias 还不够；跨 hand family 时，连续几何关系必须进入 message content，而不只是 attention score。

这两句话都成立，但它们对应的实验设计、消融重点、实现代价都不一样。

### 三、我现在的判断

如果你问“哪条更像第一篇 paper 的稳路线”，我会说：
- **A 更稳**，因为它已经足够构成一篇完整工作：`GET-style backbone + richer embodiment stream + hand-family benchmark`；
- **B 更强**，因为它更像真正的方法创新，而不是 feature completion。

所以现在最像样的选择，不是简单的 A 或 B 二选一，而是：

$$
\text{paper-1 primary claim} \in \{\text{stronger embodiment stream},\; \text{stronger relational message}\}.
$$

一旦这句话定了，后面的辅助 loss、训练范式、baseline 和实验优先级都会自然收缩。

**小结**：精读把讨论压缩成了一个很具体的选择：**我们是把贡献放在“更强的 embodiment token / static stream”，还是放在“更强的 relation message”。**

**待确认**：你更想把第一篇 paper 的主贡献落在哪一句 story 上？

## 5.3 Cross-Attention 里“cross”的准确含义：跨模态，不是跨全体 URDF 检索

**用户反馈**：用户基本拍板主体架构为方案 B，但对 `Cross-Attention(URDF, joint_state)` 的“cross”含义产生核心疑问：到底是
- A. 当前 joint state 去**所有训练过的 URDF 库**里检索；
- B. 只对**当前 episode 这只手**的 URDF 做 cross-attention。

用户判断 A 很不合理，想进一步确认：如果采用 B，跨手型泛化能力到底从何而来？GET-Zero 和 T(R,O) 的代码实际上又是怎么处理的？

**分析**：

### 一、先给结论

正确设计是 **理解 B**，但要把它说得更准确一点：

> **cross 不是“跨所有手型检索”，而是“当前样本内的 dynamic stream 与 current embodiment 的 static stream 做跨注意力”。**

也就是对第 $b$ 个样本，真正发生的是：
$$
H^{(b)} = \mathrm{CrossAttn}\big(Q=D^{(b)},\; K=S^{(b)},\; V=S^{(b)}\big),
$$
其中：
- $D^{(b)} \in \mathbb{R}^{J_b \times d}$：当前这只手在当前时刻的 joint-state / action-history tokens；
- $S^{(b)} \in \mathbb{R}^{J_b \times d}$：**同一只手** 的 URDF-derived static tokens（joint limits、rest pose、link geometry 等）；
- 注意力矩阵大小只是 $J_b \times J_b$，不是“当前 joint state 对全数据集所有 URDF”的大检索矩阵。

所以这里的 cross，**跨的是 two streams / two modalities**，不是跨整个训练集的手型库。

### 二、为什么理解 A 不对

如果把所有训练手型的 URDF 都一起塞进 Key / Value，让当前状态自己去“查哪只手更像我”，会立刻出现四个问题：

1. **推理时无法 zero-shot 到未见手型**
   - 未见 URDF 不在“记忆库”里，你就只能在旧手型里硬匹配；
   - 这更像 retrieval / nearest-neighbor，不像 embodiment-conditioned policy。

2. **语义上搞错了 conditioning 的方向**
   - 我们的问题不是“当前观测属于哪只已知手”，而是“给定当前这只手的结构，当前状态该怎么控制”。

3. **计算上没有必要**
   - 一个 batch 里如果有 10 种手，每个样本都对 10 组 URDF tokens 做 cross-attention，复杂度会无意义地膨胀；
   - 而且绝大多数 Key / Value 对当前样本根本是无关信息。

4. **会削弱方法叙事**
   - A 更像在做“hand retrieval”；
   - 我们想讲的是“shared policy conditioned on current embodiment”。

所以 A 不是一个更强的方案，而是方向就偏了。

### 三、GET-Zero 实际上怎么做：它从来没有“跨所有 URDF 检索”

GET-Zero 虽然没有显式的 `dynamic × static cross-attention`，但它处理 embodiment 的逻辑很清楚：**每个样本只带自己的 embodiment 信息**。

#### 1. 当前样本只拿当前的 embodiment ID

在 `get_zero/get_zero/distill/models/embodiment_transformer.py` 里：
- `forward(self, obs, embodiment_ids, ...)` 直接把 `embodiment_ids` 作为输入（`:162`）；
- `dof_counts = self.dof_counts_by_id[embodiment_ids]`（`:186`）说明 batch 里的每个样本都只查自己的 DoF 配置；
- `encoder_kwargs['embodiment_ids'] = embodiment_ids`（`:244`）再把它传给图编码器。

也就是说，GET 的逻辑是：
$$
\pi_\theta(a_t \mid o_t, E_b),
$$
其中 $E_b$ 是**当前样本对应手型**的 graph / embodiment 信息，而不是全体训练手型的集合。

#### 2. 图偏置也是按当前样本的 embodiment ID 取的

在 `get_zero/get_zero/distill/models/embodiment_attention.py` 里：
- `_sa_block(...)` 先拿 `unique_embodiment_ids`（`:409`）；
- 然后只索引这些样本对应的 `spd_matrices_by_id`（`:417`）、`parent_matrices_by_id`（`:426`）、`child_matrices_by_id`（`:437`）、`edge_matrices_by_id`（`:449-450`）。

这说明 GET 的 graph bias 是：
> **当前 batch 中第 $b$ 个样本，用第 $b$ 个手型自己的拓扑矩阵。**

不是“当前 token 去所有 hand graph 里搜一遍”。

### 四、T(R,O) 实际上怎么做：也是当前 robot 对当前 sample 条件化

T(R,O) 虽然不是 policy 网络，但它对 embodiment 的处理逻辑和上面完全同方向：**当前样本只用当前 robot 的结构。**

在 `TRO-Grasp/model/tro_graph.py` 里：
- 初始化时就对每个 `hand_name` 建 `create_hand_model(hand_name)`（`:86`）；
- 训练前向时逐样本取 `robot_name = batch["robot_name"][b]`（`:505, :520, :545`）；
- 再按这个 `robot_name` 取 `self.robot_links[robot_name]`、`self.link_embeddings[robot_name]`（`:506, :521` 等）。

也就是说，T(R,O) 的图构建也是：
$$
G^{(b)} = G\big(\text{current object}^{(b)},\; \text{current robot}^{(b)}\big),
$$
而不是“当前 robot 去所有机器人库里检索最像自己的 hand”。

所以无论 GET 还是 TRO，**都没有做你担心的那种多-URDF 检索式 cross-attention。**

### 五、如果采用理解 B，跨手型泛化能力到底从何而来？

不是因为“Transformer 天然支持变长输入”这么简单。**变长只是必要条件，不是核心机制。**

真正的泛化来源是四件事叠加：

1. **共享参数（shared weights）**
   - 同一套 token encoder、cross-attn、self-attn、policy head，在多种手型上共同训练；
   - 模型学到的是一个函数 $f(o_t, E)$，而不是每种手各训一套网络。

2. **显式 embodiment conditioning**
   - 当前手的 URDF / joint limits / rest pose / link geometry 直接作为输入条件；
   - 所以 unseen hand 不是“没有标签的陌生类别”，而是“带着结构描述的新样本”。

3. **结构归纳偏置（graph / relation prior）**
   - GET 用的是离散拓扑 bias；
   - 你现在拍板的 B，会进一步让连续几何关系进入 message。

4. **joint-level tokenization + per-joint action head**
   - 这让“同一个控制规律”可以在不同 DoF、不同 finger layout 上复用；
   - 如果一开始就把动作压成 hand-level 单向量，跨手型会难很多。

更准确地说，我们希望学到的是：
$$
a_t = \pi_\theta\big(o_t^{dyn},\; E^{stat},\; \mathcal{G}(E)\big),
$$
其中 $E^{stat}$ 是当前手的静态 embodiment 描述，$\mathcal{G}(E)$ 是由它诱导出来的拓扑 / 几何关系。

### 六、所以更合适的架构设计是什么

我的建议是把这一块明确写成 **sample-wise conditioning**，不要再用会让人联想到“全库检索”的表述。

#### 推荐设计

1. **当前 hand 的 static tokens**（只取当前 episode 这只手）
   - 每个 joint / link 一个 static token：
$$
s_j = \mathrm{MLP}_{stat}\big(q_{\min}, q_{\max}, T_j^{rest}, \text{axis}_j, g(\text{link}_j)\big).
$$
   - 这里的 $g(\text{link}_j)$ 可以是 BPS、bbox、尺度、主轴等几何编码。

2. **当前 hand 的 dynamic tokens**（同一只手、当前时刻）
$$
d_j = \mathrm{MLP}_{dyn}(q_j, \dot q_j, a_{t-1,j}, h_j^{hist}).
$$

3. **Cross-Attention 只在当前样本内做**
$$
h_j^{(0)} = \mathrm{CrossAttn}(Q=d_j,\; K=S,\; V=S).
$$
这里的含义不是“joint $j$ 只看自己的静态 token”，而是：
> 当前 joint $j$ 的动态状态，可以查询**同一只手的全部静态结构信息**。

这其实是合理的，因为拇指当前该怎么动，不只取决于“拇指自己的 limit”，也取决于整只手的 finger layout、其他指的结构余量、thumb opposition 的空间关系。

4. **Self-Attention / Relational Attention 仍然只在当前 hand 内做**
$$
h_i^{(1)} = \sum_j \alpha_{ij} \tilde V_{ij},
$$
其中：
- 方案 A：$\tilde V_{ij}=V_j$；
- 方案 B：$\tilde V_{ij}=h_\psi(h_i,h_j,e_{ij})$。

这里的 $e_{ij}$ 也只来自**当前 hand** 的 pairwise relation（SPD、parent-child、rest-pose relative SE(3)、link geometry pair）。

### 七、工程上怎么落地

- **如果走 GET-Zero 式 BC / distillation**：
  - batch 内可以混多手型；
  - 但每个样本自己的 cross-attn 仍然只对自己的 static tokens 做；
  - 需要 padding + mask。

- **如果走端到端 RL、每种手型一个 env group**：
  - 每次 forward 里 batch 内通常是同一种手；
  - 更简单，甚至可以不需要 padding；
  - 但 sample-wise conditioning 的语义完全不变。

所以**是否混 batch**只是实现问题，不改变 cross-attention 的定义。

**小结**：
- 正确理解是 **B**，不是 A；
- 这里的 cross 是“当前样本内，dynamic joint stream × current-hand static embodiment stream”；
- 跨手型泛化来自 shared weights + explicit embodiment conditioning + graph/relation prior，而不是“把所有 URDF 一起喂进去让网络自己检索”。

**待确认**：在这个澄清基础上，你下一步更想继续讨论哪一块：`static token 里到底放哪些 URDF / 几何量`，还是 `方案 B 里的 relation edge e_{ij} 具体由哪些量组成`？

## 5.4 Cross-Attention 是否成立，取决于 static stream 不是单个全局向量

**用户反馈**：继续追问 cross-attention 的合理性：如果 URDF 只是一个全局静态描述，那么 K / V 只有一个全局向量时，所有 joint query 都 attend 到同一个 memory，似乎没有必要用 cross-attention。用户希望明确：到底该把 static stream 做成 per-joint tokens，还是换成 concat / FiLM 等更直接的结构。

**分析**：

### 一、核心结论

这里最关键的一句话是：

> **如果 static stream 只有一个 global hand embedding，就不建议用 cross-attention；如果 static stream 是 per-joint / per-link token set，cross-attention 就成立且有意义。**

也就是说，问题不在“cross-attention 这个算子对不对”，而在：

> **URDF 不应该被压成单个全局向量。**

### 二、URDF 本来就不是只能给一个全局向量

URDF 解析后天然就是一张图，而不是一个单点描述。它至少会给出：
- 每个 joint 的 axis、limit、origin、parent / child 关系；
- 每个 link 的几何、惯性、尺寸、碰撞体；
- 整体的 kinematic tree。

所以更自然的不是：
$$
\text{URDF} \to z_{global} \in \mathbb{R}^d,
$$
而是：
$$
\text{URDF} \to \{s_1, \dots, s_J\}, \quad s_j \in \mathbb{R}^d.
$$

这其实和 GET-Zero、T(R,O) 的处理方向是一致的：
- **GET-Zero** 在论文 Sec. III-A 里本来就是 per-joint token，固定局部量是 $o_{fl,j}$；代码里也是 `global_tokens` 复制到每个关节，再和 `local_tokens` 拼成 joint tokens（`get_zero/get_zero/distill/models/embodiment_transformer.py:221-222`），不是把 URDF 压成一个全局 code 再喂给所有 joint。
- **T(R,O)** 也是 object patch nodes + link nodes，不是一个 object global vector 去管所有节点。

所以更标准的做法，应该是：
> **让 static stream 本身就是一个结构化 token 集，而不是单个全局 memory。**

### 三、那 K / V 应该具体是什么

我建议把 static stream 明确定义成：
$$
S = [s_1, s_2, \dots, s_J, s_{hand}],
$$
其中 $s_j$ 是 **当前这只手第 $j$ 个 joint / attached-link 的静态 token**，$s_{hand}$ 是可选的 hand-level summary token。

#### 每个 joint static token $s_j$ 的候选组成

最小可行版本可以是：
$$
s_j = \mathrm{MLP}_{stat}(q_{\min}, q_{\max}, a_j, T_{p\to j}^{rest}, g(l_j), d_j^{topo}),
$$
其中：
- $q_{\min}, q_{\max}$：该关节的活动范围；
- $a_j$：joint axis / joint type；
- $T_{p\to j}^{rest}$：该 joint 相对 parent 的 rest-pose 变换；
- $g(l_j)$：该 joint 所连 link 的几何编码（BPS 或轻量版 bbox / scale / PCA axes）；
- $d_j^{topo}$：拓扑离散量，如 depth、parent count、child count、finger id。

如果后面真要跨 hand family，更强一点的版本还可以加：
- actuator effort / stiffness / damping；
- link inertia / mass；
- fingertip type / contact patch descriptor。

#### hand-level summary token $s_{hand}$（可选）

它不是必须，但可以作为补充：
$$
s_{hand} = \mathrm{Pool}(s_1, \dots, s_J),
$$
用来携带：
- finger count / DoF count；
- thumb opposition 之类的全手级信息；
- overall span / palm size 等 coarse morphology。

关键点是：
> `s_hand` 只能是补充，不能替代整套 per-joint static tokens。

### 四、这样一来，cross-attention 在“cross”什么

此时动态流和静态流分别是：
$$
D = [d_1, \dots, d_J], \quad S = [s_1, \dots, s_J, s_{hand}].
$$

然后做：
$$
H = \mathrm{CrossAttn}(Q=D, K=S, V=S).
$$

这里的物理意义就变得很清楚：
- 第 $i$ 个 dynamic joint token 不是去查“哪只手最像我”；
- 而是在**当前这只手内部**，查询“我和哪些静态结构信息最相关”。

这允许：
- 拇指当前状态去读其他手指的 joint limits / finger layout；
- 食指当前状态去读 thumb opposition 对应的几何约束；
- 某个 distal joint 去读手掌 summary token。

所以它不是多手型检索，而是：
> **state-conditioned structure retrieval within the current hand.**

### 五、如果只想做局部 conditioning，确实可以不用 cross-attention

如果你最后发现自己真正需要的只是：
- 每个 joint 只看自己的 static 属性；
- 不需要在 stage-1 就让 joint 去查整只手的其他静态 token；

那确实没必要硬用 cross-attention。更直接的替代是：

#### 方案 1：concat
$$
\tilde d_j = \mathrm{MLP}([d_j, s_j]).
$$

#### 方案 2：FiLM
$$
\tilde d_j = \gamma(s_j) \odot d_j + \beta(s_j).
$$

这两种都比“单 global token 做 cross-attention”更合理。

### 六、所以更像样的折中设计是什么

我觉得目前最干净的折中是：

1. **先做 per-joint local conditioning**
   - 用 concat 或 FiLM 先把 $s_j$ 融到对应的 $d_j$；

2. **再做 structured cross-attention**（如果你仍想保留这个 stage）
   - Query = 已被 local-conditioned 的 dynamic tokens；
   - Key / Value = 当前手的全体 static tokens $S$；

3. **最后进方案 B 的 relation self-attention**
   - 把真正的 pairwise relation 留给 stage-2。

写成公式就是：
$$
\hat d_j = \gamma(s_j) \odot d_j + \beta(s_j),
$$
$$
h^{(0)} = \mathrm{CrossAttn}(\hat D, S, S),
$$
$$
h^{(1)}_i = \sum_j \alpha_{ij} h_\psi(h_i, h_j, e_{ij}).
$$

这样有两个好处：
- **cross-attention 不再悬空**：因为 K / V 确实来自 structured static token set；
- **stage-1 / stage-2 分工更清楚**：stage-1 做 state–structure 对齐，stage-2 做 relation message passing。

### 七、当前判断

所以我不建议你把 cross-attention 整个推翻，但我会强烈建议你**把 static stream 从“global URDF embedding”改成“per-joint static tokens + optional hand summary token”**。

若你坚持只保留一个 global URDF code，那么我反而会建议：
> **别用 cross-attention，直接改成 FiLM / concat。**

**小结**：
- 单个全局 URDF 向量 → 不值得做 cross-attention；
- per-joint static tokens（可外加一个 hand summary token） → cross-attention 合理，而且更接近 GET / TRO 的结构化表达；
- 因此，当前最合理的改动不是“放弃 cross-attention”，而是**重新定义 K / V 的来源**。

**待确认**：下一步你要我先把 `per-joint static token` 的字段表列细，还是把 `relation edge e_{ij}` 的字段表列细？

## 5.5 当前拍板版的最终数据流图（sample-wise conditioning 版本）

**用户反馈**：希望把当前拍板版架构画成最终数据流图，并用 LaTeX 公式注解。

**分析**：

下面这张图对应我们在 5.3 / 5.4 中已经拍板的前提：
- **不是**“当前状态去所有 URDF 库里检索”；
- 而是**当前样本内**，dynamic joint stream 查询 **当前这只手** 的 structured static tokens；
- stage-2 再进入方案 B 的 relation-conditioned self-attention。

```mermaid
flowchart LR
    U["Current-hand URDF / MJCF"] --> SA[Per-joint static attrs]
    G["Link geometry BPS or bbox/PCA"] --> SA
    SA --> SE["Static token encoder<br/>S = {s_1 ... s_J, s_hand}"]

    O["Joint state / prev action / history"] --> DE["Dynamic token encoder<br/>D = {d_1 ... d_J}"]
    DE --> LC["Local conditioning<br/>FiLM or concat"]
    SE --> LC

    LC --> CA["Cross-Attention<br/>Q = D_hat, K = S, V = S"]
    SE --> CA

    SE --> EG["Relation edge builder<br/>e_ij"]
    CA --> RA["Relational Self-Attention<br/>edge-conditioned value"]
    EG --> RA

    RA --> PH[Per-joint policy head]
    PH --> A[Joint-space action]
```

### 一、Stage-0：把当前手的 URDF 解析成 structured static tokens

这里最关键的是：**K / V 不是一个 global URDF 向量，而是一组当前手的 static tokens。**

对第 $j$ 个 joint，我们定义：
$$
s_j = \mathrm{MLP}_{stat}\Big(q_{\min,j}, q_{\max,j}, a_j, T_{p\to j}^{rest}, g(l_j), d_j^{topo}\Big),
$$
其中：
- $q_{\min,j}, q_{\max,j}$：joint limits；
- $a_j$：joint axis / joint type；
- $T_{p\to j}^{rest}$：该 joint 相对 parent 的 rest-pose 变换；
- $g(l_j)$：attached link 的几何编码（BPS 或轻量几何描述）；
- $d_j^{topo}$：depth / parent count / child count / finger id 等拓扑离散量。

如果需要一个 hand-level summary token，可再定义：
$$
s_{hand} = \mathrm{Pool}(s_1, \dots, s_J).
$$

### 二、Stage-1：dynamic joint tokens 与 static tokens 的对齐

对第 $j$ 个 joint 的动态流，我们定义：
$$
d_j = \mathrm{MLP}_{dyn}(q_j, \dot q_j, a_{t-1,j}, h_j^{hist}).
$$

在进入 cross-attention 之前，先做一层局部 conditioning（可选但我认为合理）：

#### 版本 1：FiLM
$$
\hat d_j = \gamma(s_j) \odot d_j + \beta(s_j),
$$

#### 版本 2：concat
$$
\hat d_j = \mathrm{MLP}_{cond}([d_j, s_j]).
$$

然后做当前样本内的 structured cross-attention：
$$
H^{(0)} = \mathrm{CrossAttn}(\hat D, S, S),
$$
其中：
$$
\hat D = [\hat d_1, \dots, \hat d_J], \qquad S = [s_1, \dots, s_J, s_{hand}].
$$

这里的含义不是“joint $i$ 去查哪只手最像我”，而是：
> joint $i$ 的当前状态，在**当前这只手**的全部静态结构信息中查询与自己最相关的部分。

### 三、Stage-2：方案 B 的 relation-conditioned self-attention

对于 relation edge，我们不再只保留离散 bias，而是显式构造：
$$
e_{ij} = \Big[\phi^{SPD}(i,j),\; \phi^P(i,j),\; \phi^C(i,j),\; \Delta T_{ij}^{rest},\; \Delta g_{ij}\Big].
$$

其中：
- $\phi^{SPD}(i,j)$：无向最短路径距离；
- $\phi^P(i,j), \phi^C(i,j)$：有向 parent / child 关系；
- $\Delta T_{ij}^{rest}$：rest pose 下的相对 SE(3)；
- $\Delta g_{ij}$：link-pair 几何关系（可显式，也可由 MLP 隐式组合）。

然后做方案 B：
$$
Q_i = W_Q h_i^{(0)}, \qquad K_j = W_K h_j^{(0)},
$$
$$
\alpha_{ij} \propto \exp\left(\frac{Q_i K_j^T}{\sqrt d} + b^{topo}_{ij} + b^{geom}_{ij}\right),
$$
$$
\tilde V_{ij} = h_\psi\big(h_i^{(0)}, h_j^{(0)}, e_{ij}\big),
$$
$$
h_i^{(1)} = \sum_j \alpha_{ij} \tilde V_{ij}.
$$

这里的核心点是：
> 连续几何关系不只改“看谁更多”，还改“传什么消息”。

### 四、Stage-3：per-joint policy head

最后输出 joint-space action：
$$
a_j = \pi_j\big(h_j^{(1)}\big).
$$

如果想保留 earlier rounds 的“local 主路 + global 残差”写法，也可以写成：
$$
a_j = \pi_{local}(h_j^{(1)}) + \Delta \pi_{global}(h_j^{(1)}, s_{hand}).
$$

### 五、这一版图的真正含义

这张图把几个容易混淆的点彻底分开了：

1. **URDF 不是单个 global code**，而是一组 structured static tokens；
2. **cross-attention 不做 hand retrieval**，只做当前样本内的 state–structure alignment；
3. **relation self-attention 才是方案 B 的主创新位置**，负责把 pairwise 几何关系写进 message；
4. **joint-space head 保留不动**，和你的动作空间原则一致。

### 六、如果未来放弃 cross-attention，该怎么退化

如果你后面发现 stage-1 的 cross-attention 太重或收益不够，这张图还能自然退化为：
$$
\hat d_j = \gamma(s_j) \odot d_j + \beta(s_j),
$$
$$
h_i^{(1)} = \mathrm{RelSelfAttn}(\hat D, e_{ij}),
$$
也就是 **FiLM / concat + 方案 B self-attn**。

所以当前图不是把自己锁死，而是保留了一个很自然的降阶版本。

**小结**：当前最合理的最终数据流图是：
$$
\text{URDF} \to \text{structured static tokens} \to \text{local conditioning} \to \text{cross-attn} \to \text{relation self-attn} \to \text{per-joint action}.
$$
这已经把“cross 在哪、relation 在哪、action 在哪”三件事分得很清楚了。

**待确认**：在这张总图上，你下一步更想让我把 `per-joint static token` 字段表列细，还是把 `relation edge e_{ij}` 字段表列细？

## 5.6 信息流与残差：当前图里真正缺的不是“更多 token”，而是明确的 local bypass

**用户反馈**：担心 `Relational Self-Attention -> per-joint policy head` 太直接，可能洗掉原始 joint state 的细粒度信息；提出是否需要把 dynamic token encoder 的输出与最终隐状态做 ResNet 式残差拼接，并希望结合 TRO-Grasp / GET-Zero 的成功设计重新审视。

**分析**：

### 一、先给结论

你的直觉是对的，但要分两层看：

1. **`final hidden state -> head` 本身不怪**
   - GET-Zero 其实就是这么干的：`tokens = self.encoder(**encoder_kwargs)`（`get_zero/get_zero/distill/models/embodiment_transformer.py:246`），然后 `head_output = head_module(tokens)`（`:256`）。
   - 所以“head 直接吃最终隐状态”不是原罪。

2. **真正的前提是：attention block 内部必须是 residual 更新，而不是覆盖式替换**
   - GET 在编码器层里明确用了残差：`x = x + self._sa_block(...)`、`x = x + self._ff_block(...)`（`get_zero/get_zero/distill/models/embodiment_attention.py:373-377`）。
   - TRO 在 OR / RR 里也都有显式 self residual：`robot_node_f = agg_or + self_or`（`TRO-Grasp/model/denoiser.py:258`）、`robot_node_f = agg_rr + self_rr`（`:297`）。

所以如果我们当前的高层图被实现成：
$$
d \to \text{CrossAttn} \to \text{RelSelfAttn} \to \text{Head},
$$
而**没有** residual / FFN / local bypass，那我同意你的担心：这会形成一个不必要的信息瓶颈。

### 二、我对当前架构的专业判断

如果要把它变成一个更稳、更像成功先例的结构，我会明确补上三层机制：

#### 1. Block-level residual（必须）

Cross-attention 和 relation self-attention 都不应当“覆盖”输入，而应当做：
$$
h^{ca} = \hat D + \mathrm{CrossAttn}(\mathrm{LN}(\hat D), \mathrm{LN}(S), \mathrm{LN}(S)),
$$
$$
h^{ca} = h^{ca} + \mathrm{FFN}(\mathrm{LN}(h^{ca})),
$$
$$
h^{rel} = h^{ca} + \mathrm{RelSelfAttn}(\mathrm{LN}(h^{ca}), e),
$$
$$
h^{rel} = h^{rel} + \mathrm{FFN}(\mathrm{LN}(h^{rel})).
$$

也就是说，最终隐状态不是“被洗出来的新东西”，而是：
> **原始动态特征 + 结构对齐增量 + 关系通信增量。**

#### 2. Head-level local bypass（我认为也应该加）

即使 block 内部有 residual，我仍然建议在 action head 前保留一条显式 local 主路。理由很简单：
- per-joint control 的低层精度，本来就高度依赖当前关节的 proprioception；
- relation branch 更适合学“修正量 / 协调量”，而不是独自承担全部动作生成。

所以更像样的 action 写法是：
$$
a_j^{local} = \pi_{local}(\hat d_j),
$$
$$
\Delta a_j^{rel} = \pi_{rel}([\hat d_j, h_j^{rel}]),
$$
$$
a_j = a_j^{local} + \Delta a_j^{rel}.
$$

这里：
- $\hat d_j$：local-conditioned dynamic token；
- $h_j^{rel}$：经过 structure alignment + relational message passing 后的上下文化隐状态。

这其实就是我们第二轮里提过的 **local 主路 + global / relational residual**，只是现在把它安到了更明确的位置上。

#### 3. 最好让 relational branch 也能看到 local token（而不只看 final hidden）

如果你只让 relational head 看 $h_j^{rel}$，它虽然理论上已经包含了 local 信息，但表达上仍然不够保险。

所以我更推荐：
$$
\Delta a_j^{rel} = \pi_{rel}([\hat d_j, h_j^{rel}]),
$$
而不是单独用 $h_j^{rel}$。

原因是：
- 这相当于在 head 前再给它一次“原始局部状态 + 全局关系上下文”的并置；
- 更利于训练时学出“基于 local state 的小修正”，而不是强迫 relational branch 重建 local state。

### 三、从信息流角度看，当前设计哪里确实容易出问题

如果我们**不**加上面这三点，潜在问题主要有两个：

1. **局部可逆信息被 attention 混合后难以显式恢复**
   - joint angle、velocity、prev action 这种低层控制量，本来是最容易直接映射到 action 的；
   - 若一股脑送进两层 attention 再单头输出，网络当然“理论上能学回来”，但训练上没必要给它增加这个负担。

2. **梯度路径会过长，identity mapping 不够容易**
   - 残差连接的一个核心价值就是：如果当前 relation module 没学到有用东西，网络至少能退回近似恒等映射；
   - 没有残差时，模型被迫每层都重新编码，会更不稳。

### 四、因此我会把当前总图改成下面这个版本

#### 推荐版公式

先做局部 conditioning：
$$
\hat d_j = \gamma(s_j) \odot d_j + \beta(s_j)
\quad \text{或} \quad
\hat d_j = \mathrm{MLP}_{cond}([d_j, s_j]).
$$

再做带残差的 cross-attention block：
$$
H^{ca} = \hat D + \mathrm{CrossAttn}(\mathrm{LN}(\hat D), \mathrm{LN}(S), \mathrm{LN}(S)),
$$
$$
H^{ca} = H^{ca} + \mathrm{FFN}(\mathrm{LN}(H^{ca})).
$$

再做带残差的 relation self-attention block：
$$
H^{rel} = H^{ca} + \mathrm{RelSelfAttn}(\mathrm{LN}(H^{ca}), e),
$$
$$
H^{rel} = H^{rel} + \mathrm{FFN}(\mathrm{LN}(H^{rel})).
$$

最后用 local 主路 + relational residual 输出动作：
$$
a_j = \pi_{local}(\hat d_j) + \Delta\pi_{rel}([\hat d_j, h_j^{rel}]).
$$

### 五、如果还想更稳，可以再加一个门控

如果担心训练初期 relational branch 乱改动作，可以再加一个 gate：
$$
a_j = a_j^{local} + \lambda_j \odot \Delta a_j^{rel},
$$
$$
\lambda_j = \sigma(W h_j^{rel}).
$$

这样模型可以在 early stage 先主要走 local branch，等 relation 学稳了再逐渐放大修正量。

### 六、所以我们当前架构还缺什么

若按“成熟深度学习骨架”来审视，当前图里最该补的是：

1. **Cross-Attn block 的 residual + FFN**
2. **Relational Self-Attn block 的 residual + FFN**
3. **Policy head 前的 local bypass**
4. **最好让 relation branch 也显式看到 local token**
5. **（可选）一个 gate 控制 relational residual 的幅度**

注意：这里缺的不是更多 token，而是**信息保真路径**。

### 七、我的最终判断

如果你问我作为深度学习研究者，这个架构最值得怎么改，我的回答会很明确：

> **不要把“最终隐状态直接接 head”简单理解成问题；真正的问题是高层图里没有把 residual / FFN / local bypass 画出来。把这些补上后，这个架构会合理很多。**

也就是说，我不建议推翻当前架构，而建议把它改成：

$$
\text{local-conditioned dynamic tokens}
\to
\text{residual cross-attn block}
\to
\text{residual relational self-attn block}
\to
\text{local main path + relational residual head}.
$$

**小结**：
- 你的担心是成立的；
- 但问题不在“attention 输出不能接 head”，而在**需要明确的信息保真与残差路径**；
- 当前最合理的修正，是把 `local 主路 + relational residual` 明确写回架构里。

**待确认**：你下一步更想让我把这个“带 local bypass 的修正版总图”重新画出来，还是把 `policy head` 的具体输入 / 输出形式（concat、残差相加、gate）列成对比表？

## 5.7 修正版总图：Residual Cross-Attn + Residual Relational Self-Attn + Local Bypass

**用户反馈**：希望把 5.6 里提出的修正版真正重画成一张总图，把 residual、FFN、local bypass 和 relational residual 都明确画出来。

**分析**：

下面这张图是我现在认为**更稳、更像成功先例**、也更适合你当前任务边界的版本。

```mermaid
flowchart LR
    U["Current-hand URDF / MJCF"] --> SA["Per-joint static attrs"]
    G["Link geometry<br/>BPS or bbox/PCA"] --> SA
    SA --> ST["Structured static tokens<br/>S = s_1 ... s_J, s_hand"]

    O["Joint state / prev action / history"] --> DY["Dynamic token encoder<br/>D = d_1 ... d_J"]
    DY --> LC["Local conditioning<br/>FiLM or concat"]
    ST --> LC

    LC --> LM["Local main path<br/>a_local"]

    LC --> CA["Residual Cross-Attn block<br/>+ FFN"]
    ST --> CA

    ST --> EG["Relation edge builder<br/>e_ij"]
    CA --> RA["Residual Relational Self-Attn block<br/>+ FFN"]
    EG --> RA

    LC --> RH["Relational residual head<br/>Input = d_hat_j, h_rel_j"]
    RA --> RH

    LM --> SUM["Sum / Gated Sum"]
    RH --> SUM
    SUM --> A["Joint-space action"]
```

### 一、这张图相对 5.5 的核心改动

相较于上一版，这里明确补了四件事：

1. **Cross-Attn 不是裸层，而是 residual block + FFN**；
2. **Relational Self-Attn 也不是裸层，而是 residual block + FFN**；
3. **Policy 输出前显式保留一条 local main path**；
4. **Relational residual head 不是只看 final hidden，而是同时看 $\hat d_j$ 和 $h_j^{rel}$。**

这四点合起来，才更像一个成熟的深度学习架构，而不是“把所有信息扔进 attention 黑盒再 hoping for the best”。

### 二、按公式写，就是下面这套

#### 1. 动态流与静态流

先定义动态 token 和静态 token：
$$
d_j = \mathrm{MLP}_{dyn}(q_j, \dot q_j, a_{t-1,j}, h_j^{hist}),
$$
$$
s_j = \mathrm{MLP}_{stat}(q_{\min,j}, q_{\max,j}, a_j, T_{p\to j}^{rest}, g(l_j), d_j^{topo}).
$$

局部 conditioning 后得到：
$$
\hat d_j = \gamma(s_j) \odot d_j + \beta(s_j)
\quad \text{或} \quad
\hat d_j = \mathrm{MLP}_{cond}([d_j, s_j]).
$$

#### 2. Residual Cross-Attn block

$$
H^{ca} = \hat D + \mathrm{CrossAttn}(\mathrm{LN}(\hat D), \mathrm{LN}(S), \mathrm{LN}(S)),
$$
$$
H^{ca} = H^{ca} + \mathrm{FFN}(\mathrm{LN}(H^{ca})).
$$

这一步的作用是：
> 让当前 joint state 在**当前手的结构化静态信息**中做查询，但不丢掉原始动态主干。

#### 3. Residual Relational Self-Attn block

先构 relation edge：
$$
e_{ij} = \big[\phi^{SPD}(i,j),\; \phi^P(i,j),\; \phi^C(i,j),\; \Delta T_{ij}^{rest},\; \Delta g_{ij}\big].
$$

然后做方案 B：
$$
H^{rel} = H^{ca} + \mathrm{RelSelfAttn}(\mathrm{LN}(H^{ca}), e),
$$
$$
H^{rel} = H^{rel} + \mathrm{FFN}(\mathrm{LN}(H^{rel})).
$$

这里 Relational Self-Attn 内部可以写成：
$$
\alpha_{ij} \propto \exp\left(\frac{Q_iK_j^T}{\sqrt d} + b^{topo}_{ij} + b^{geom}_{ij}\right),
$$
$$
\tilde V_{ij} = h_\psi(h_i^{ca}, h_j^{ca}, e_{ij}),
$$
$$
h_i^{rel} = \sum_j \alpha_{ij} \tilde V_{ij}.
$$

#### 4. Local main path + relational residual head

局部主路：
$$
a_j^{local} = \pi_{local}(\hat d_j).
$$

关系修正路：
$$
\Delta a_j^{rel} = \pi_{rel}([\hat d_j, h_j^{rel}]).
$$

最终动作：
$$
a_j = a_j^{local} + \lambda_j \odot \Delta a_j^{rel},
$$
其中：
$$
\lambda_j = 1
\quad \text{(无门控)}
\qquad \text{或} \qquad
\lambda_j = \sigma(W h_j^{rel})
\quad \text{(有门控)}.
$$

### 三、为什么这版比“直接 final hidden -> head”更合理

#### 1. 原始局部控制量不会被迫在 attention 里“绕一圈再学回来”

如果没有 local main path，网络必须让 relation branch 同时承担：
- 低层 proprio 控制；
- embodiment conditioning；
- 全局协调。

这当然不是做不到，但训练上没有必要这么拧巴。

#### 2. relation branch 的职责被压缩成“修正量 / 协调量”

这更符合它的长板：
- 它擅长整合跨 joint 的结构关系；
- 不一定最适合独自生成全部低层动作。

#### 3. 残差让 identity mapping 更容易

这点 GET 和 TRO 都已经给出经验支持：
- GET 的 Transformer 层靠残差保持信息与梯度；
- TRO 的 OR / RR 都靠 `agg + self` 保留节点自身信息。

所以我们的结构也应当把这种“保底通路”明确画出来。

### 四、这版图和 earlier rounds 的一致性

它其实没有推翻我们前面的结论，反而是把第二轮就出现过的那句：
> **local 主路 + global / relational residual**

重新落回到了一个更精确的位置上。

也就是说，现在整体架构可以被概括成：
$$
\text{local-conditioned dynamic tokens}
\to
\text{residual cross-attn block}
\to
\text{residual relational self-attn block}
\to
\text{local main path + relational residual action head}.
$$

### 五、如果后面想再简化，也很自然

如果实验后发现 cross-attn 收益不大，这张图还能平滑退化为：
$$
\hat d_j = \mathrm{LocalCond}(d_j, s_j),
$$
$$
H^{rel} = \mathrm{ResidualRelSelfAttn}(\hat D, e),
$$
$$
a_j = \pi_{local}(\hat d_j) + \Delta\pi_{rel}([\hat d_j, h_j^{rel}]).
$$

所以这版图不仅更稳，还保留了后续降阶空间。

**小结**：当前更成熟的总图，不应该是“attention 黑盒直出动作”，而应该是：
- **attention block 内部有 residual + FFN**；
- **动作头外部有 local bypass + relational residual**；
- **relation branch 学修正，local branch 保底层控制精度**。

**待确认**：在这张修正版总图基础上，你下一步更想让我继续细化 `policy head 三种实现（concat / residual sum / gate）`，还是回到 `static token / relation edge` 的字段表？

## 5.8 频率与时间步：GET 更像“feature-stacked short-history policy”，而不是 temporal transformer

**用户反馈**：希望回答两个工程上很关键的问题：
1. GET / Graph Embodiment Transformer 在推理阶段能否做到 `>20Hz`，这对手内操作很重要；
2. GET 到底是单时间步策略，还是对 joint token 做了多时间步编码。

**分析**：

### 一、先给结论

1. **从架构复杂度上看，GET 类网络做 `>20Hz` 是可行的；真正可能卡频率的往往不是 attention，而是硬件 I/O、Python 控制循环和感知处理。**
2. **GET 不是纯单时间步 $\pi(o_t) \to a_t$；它包含 history，但这个 history 是在 tokenization 之前沿 feature 维堆叠进去的，不是做 temporal attention。**

也就是说，GET 更准确地说是：
$$
a_t = \pi_\theta\big(\text{stacked short history features up to } t\big),
$$
而不是：
$$
a_t = \mathrm{TemporalTransformer}(o_{t-H+1}, \dots, o_t).
$$

### 二、GET 的推理路径其实很薄

在 `get_zero/get_zero/rl/models/embodiment_transformer.py` 里，RL 推理包装器基本只做了三件事：
- 取 `obs = obs_dict['obs']`；
- 构造 `embodiment_ids`；
- 调 `self.embodiment_transformer(obs, embodiment_ids, ...)` 拿 `policy` 输出。

也就是：
- `obs.size(0)` 允许**batch inference**（模拟里可并行多个 env）；
- 但部署时通常就是 batch size = 1；
- 没有 RNN rollout、没有优化求解、没有 diffusion sampling。

所以单看网络前向，它并不重。

### 三、部署代码本身就是按 20Hz 写的

在 `get_zero/get_zero/deploy/leap/deploy.py` 里：
- `hz = 20`（`:159`）
- `self.control_dt = 1 / hz`（`:160`）
- `rate = Rate(hz)`（`:161`）
- 控制循环里每步只调一次 `network_action = self.forward_network(obs_buf[0])`（`:338`）
- 并实时监控 `actual_hz_running_average`（`:403-408`），若偏离目标 20Hz 超过 3Hz 就打印 warning
- 最后每步 `rate.sleep()` 保持 20Hz（`:411`）

这说明作者在真实硬件部署时，**就是把 GET 放在 20Hz 控制环里用**。

所以如果你问“作者自己觉得它能不能 20Hz”，答案显然是：**他们就是按这个目标写的部署代码。**

### 四、复杂度上为什么 16-DoF 手不太像会卡在 attention 上

设：
- $J$ = joint token 数（LeapHand 约 16）
- $L$ = attention 层数
- $d$ = token hidden dim

单层 attention 的主要复杂度可粗略写成：
$$
\mathcal O(J d^2 + J^2 d).
$$

其中：
- $J d^2$ 来自 Q/K/V / output projection；
- $J^2 d$ 来自 attention score 与 weighted sum。

对于 LeapHand：
$$
J \approx 16 \quad \Rightarrow \quad J^2 = 256.
$$

这个量级非常小。对比之下，硬件环里真正重的通常是：
- `poll_joint_position()` 的通信开销；
- `command_joint_position()` 的通信开销；
- AR tag / 相机处理；
- Python 侧数据搬运与安全检查。

所以**网络前向本身大概率不是 20Hz 的主瓶颈**。

### 五、但我不会在这里伪造一个“实际 Hz 数字”

我们这轮没有实际跑 benchmark，所以我不会编造“在某 GPU 上一定是 143Hz”这种数字。

更谨慎、也更可信的说法应该是：
- **代码层面**：GET 部署路径就是按 20Hz 写的；
- **复杂度层面**：16-DoF 上 attention 成本很小，20Hz 非常 plausible；
- **工程层面**：实际闭环频率更可能由硬件通信和感知链路决定，而不是由 network forward 决定。

### 六、GET 到底是单时间步还是多时间步

#### 1. 标准 observation 路径：明确包含 3-step history

在 `get_zero/get_zero/rl/tasks/leap_hand_rot.py` 里：
- 若 `tokenizeObservationForPolicy=False`，环境直接设
  `numObservations = (dof_count + dof_count + 2) * 3`（`:69` 左右）
- 在 `compute_standard_observation()` 中，它维护 `standard_obs_buf_lag_history`，并取最后 3 个时间步拼成 `t_buf` 再送给策略。

所以标准 MLP policy 明确是一个 **3-step history stacked observation**。

#### 2. tokenized GET 路径：也包含 history，但方式不同

在 `compute_tokenized_observation()` 里：
- 它直接断言 `include_history` 必须为 True（`leap_hand_rot.py:596`）；
- 然后调 `self.observation_tokenizer.build_tokenized_observation(...)`（`:599`）。

而 `ObservationTokenizer` 里：
- `self.obs_history_size = tokenization_cfg['obsHistorySize']`（`get_zero/get_zero/distill/utils/embodiment_util.py:196`）；
- 默认 tokenization config 里 `obsHistorySize: 6`（`get_zero/get_zero/distill/cfg/tokenization/Default.yaml:22`）；
- 它维护 `variable_global_obs_history_buf` 和 `variable_local_obs_history_buf`；
- 再把 history 沿 feature 维 flatten 到每个 token 里（`embodiment_util.py:311-395`）。

也就是说，tokenized GET 的 history 是：
$$
\bar o_t^{g} = [o_t^{g,var}, o_{t-1}^{g,var}, \dots, o_{t-H+1}^{g,var}, o^{g,fix}],
$$
$$
\bar o_{t,j}^{l} = [o_{t,j}^{l,var}, o_{t-1,j}^{l,var}, \dots, o_{t-H+1,j}^{l,var}, o_j^{l,fix}].
$$

然后 joint token 才是：
$$
T_j = [\bar o_t^g, \bar o_{t,j}^l].
$$

所以这不是“时间维上的 token 序列”，而是：
> **历史先被堆进 feature 维，再作为单个 joint token 的输入。**

### 七、GET 在 LeapHandRot 里到底把什么放进 history

从 `LeapHandRot.get_time_varying_raw_obs()` 和 task config 看：
- variable global obs：`phase`（部署里是 `sin/cos phase`）
- variable local obs：`noisy_dof_pos` 和 `cur_targets`

也就是 tokenized GET 更接近：
$$
a_t = \pi_\theta\big(q_{t-H+1:t},\; \text{target}_{t-H+1:t},\; \phi_{t-H+1:t},\; o^{fix}\big) \to \Delta \text{target}_t.
$$

注意这里它显式保留的是 **target history**，不一定是你写的原始 `a_{t-1}`，但在 LEAP 这类 delta-target 控制里，这两者在语义上是接近的：
- action 决定 target update；
- target 又会进入下一时刻 observation。

### 八、这对我们架构的直接启发

如果你把问题换成“我们自己的 AnyMani 架构该怎么做”，我会给一个很明确的建议：

1. **若目标是 `>20Hz`，不要做 temporal attention over time axis**
   - 这会平白增加 token 数和 attention 成本；
   - GET 也没有这么做。

2. **history 要么 feature-stack，要么放到图外的小 temporal encoder**
   - GET 走的是 feature-stack；
   - 我们前面讨论的 per-joint GRU / TCN 也可以，但最好放图外。

3. **第一版完全可以沿你熟悉的单步策略稍微加一点历史**
   - 例如：$\pi(q_t, \dot q_t, a_{t-1}, s_j, e_{ij}) \to a_t$；
   - 或 feature-stack 3~6 步历史；
   - 不必一上来做时空联合 Transformer。

### 九、我的最终判断

- **频率问题**：GET 类架构在 16-DoF 手上做 20Hz 是合理目标，且部署代码本来就是按 20Hz 写的；
- **时间步问题**：GET 不是纯单步，但也不是 temporal transformer；它是**short history stacked features + spatial graph transformer**；
- **对我们最稳的路线**：把 temporal modeling 放在图外（feature stack 或小 history encoder），把图注意力留给“空间 / embodiment / relation”。

**小结**：GET 更像
$$
\text{short-history feature stack} + \text{joint-token graph transformer},
$$
而不是
$$
\text{temporal transformer} + \text{graph transformer}.
$$
这对 `>20Hz` 是友好的，也和你在 IsaacLab 里已经跑通的策略习惯并不冲突。

**待确认**：在这个结论基础上，你下一步更想让我继续讨论 `我们的历史建模该选 feature-stack、single-step+prev action，还是图外小 GRU/TCN`，还是 `我们架构的 20Hz 预算该如何粗分（network / tokenizer / env / hardware I/O）`？